# Telco Churn: da predição ao plano de retenção

**Objetivo de negócio:** identificar, entre os clientes ativos, quem tem maior probabilidade de cancelar, **priorizar** quem deve ser contatado primeiro e **estimar o retorno financeiro** das ações de retenção.

**Dataset:** Telco Customer Churn (7.043 clientes, 21 colunas, ~26,5% de churn).

### Roteiro

| Seção | O que acontece |
|---|---|
| 0 | Setup, configuração e função que salva as figuras em `./outputs` |
| 1 | Carga, limpeza, engenharia de features e pré-processamento **sem vazamento de dados** |
| 2 | Métricas adequadas (Precision, Recall, F1, F2, ROC-AUC) e tratamento do desbalanceamento (`class_weight` × SMOTE) |
| 3 | Random Forest × XGBoost × CatBoost, otimização de hiperparâmetros, threshold via F2 e Feature Importance |
| 4 | Lista de Prioridade de Risco, segmentação, plano de ação e simulação de ROI |
| 5 | Exportação dos outputs (figuras `.png`, CSVs e modelo) |

### O que muda em relação à versão anterior

- `customerID` agora é **mantido como índice**. Antes ele era descartado, e uma lista de prioridade sem identificação do cliente não serve ao time de Customer Success.
- `TotalCharges`: **uma única regra** de tratamento (sem `fillna` redundante depois de `dropna`).
- `pd.get_dummies` em toda a base antes do split foi substituído por um `ColumnTransformer` dentro de um `Pipeline`: scaler e encoder aprendem **só com o treino**.
- Acurácia deixa de ser a métrica principal. Passam a valer Precision, Recall, F1, F2, ROC-AUC e PR-AUC.
- Um Random Forest solitário vira uma comparação de 3 algoritmos × 2 técnicas de balanceamento, com busca de hiperparâmetros e threshold otimizado.
- A saída deixa de ser "0 ou 1" e passa a ser uma **lista priorizada com plano de ação e ROI**.

### Regra de ouro deste notebook
O conjunto de **teste só é usado na avaliação final**. Toda decisão (modelo, técnica de balanceamento, hiperparâmetros e threshold) é tomada apenas com dados de treino, via validação cruzada.

---
## 0. Setup e configuração

In [ ]:
!pip install -q catboost xgboost imbalanced-learn

In [ ]:
import os
import shutil
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline            # Pipeline do imblearn: aplica SMOTE só no treino de cada fold
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, classification_report, confusion_matrix,
                             fbeta_score, f1_score, make_scorer, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score, roc_curve, accuracy_score)
from sklearn.model_selection import (RandomizedSearchCV, StratifiedKFold, cross_val_predict,
                                     cross_validate, train_test_split)
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# ---------------- Parâmetros gerais ----------------
SEED = 42
N_JOBS = -1            # paralelismo na validação cruzada (cada estimador roda com 1 thread)
N_ITER_BUSCA = 25      # combinações testadas no RandomizedSearchCV (aumente para uma busca mais ampla)
OUTPUT_DIR = './outputs'

np.random.seed(SEED)
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({'figure.dpi': 110, 'axes.titleweight': 'bold'})
COR_CHURN, COR_OK, COR_DESTAQUE = '#C44E52', '#4C72B0', '#DD8452'

# ---------------- Seção 5 (parte 1): pasta de outputs + função para salvar figuras ----------------
os.makedirs(OUTPUT_DIR, exist_ok=True)      # cria ./outputs se não existir


def salvar_fig(nome, fig=None, dpi=150):
    """Salva a figura atual em ./outputs/<nome>.png. Chame ANTES de plt.show()."""
    fig = fig or plt.gcf()
    caminho = os.path.join(OUTPUT_DIR, f'{nome}.png')
    fig.savefig(caminho, dpi=dpi, bbox_inches='tight')
    print(f'[figura salva] {caminho}')
    return caminho


def fmt_pct(df, colunas, casas=1):
    """Cópia do DataFrame com as colunas formatadas em %, só para exibição."""
    saida = df.copy()
    for c in colunas:
        saida[c] = saida[c].map(lambda v: f'{v:.{casas}%}')
    return saida


print('Ambiente pronto. Outputs serão salvos em:', os.path.abspath(OUTPUT_DIR))

---
## 1. Carga, limpeza e pré-processamento (sem vazamento de dados)

### 1.1 Carga dos dados
A função abaixo tenta, nesta ordem: arquivo local → URL pública (cópia do mesmo dataset no GitHub da IBM) → Kaggle CLI → upload manual. Assim o notebook roda de ponta a ponta no Colab sem depender de credenciais do Kaggle.

In [ ]:
ARQUIVO_KAGGLE = 'WA_Fn-UseC_-Telco-Customer-Churn.csv'
ARQUIVOS_LOCAIS = [ARQUIVO_KAGGLE, 'Telco-Customer-Churn.csv']
URL_PUBLICA = ('https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/'
               'master/data/Telco-Customer-Churn.csv')


def carregar_telco():
    """Tenta arquivo local -> URL pública -> Kaggle CLI -> upload manual (Colab)."""
    for caminho in ARQUIVOS_LOCAIS:
        if os.path.exists(caminho):
            print(f'Lendo arquivo local: {caminho}')
            return pd.read_csv(caminho)
    try:
        print('Baixando o dataset da URL pública...')
        return pd.read_csv(URL_PUBLICA)
    except Exception as erro:
        print(f'  download falhou ({type(erro).__name__}); tentando Kaggle...')
    codigo = os.system(f'kaggle datasets download -d blastchar/telco-customer-churn '
                       f'-f {ARQUIVO_KAGGLE} --unzip -q')
    if codigo == 0 and os.path.exists(ARQUIVO_KAGGLE):
        return pd.read_csv(ARQUIVO_KAGGLE)
    print('Faça o upload do CSV da Telco:')
    from google.colab import files
    enviado = files.upload()
    return pd.read_csv(next(iter(enviado)))


df_raw = carregar_telco()
print(f'\nShape: {df_raw.shape}  |  customerID único: {df_raw["customerID"].is_unique}')
display(df_raw.head(3))

### 1.2 Limpeza
- `customerID` vira **índice** (não é feature, mas precisamos dele na lista de prioridade).
- `TotalCharges` vem como texto e tem 11 valores em branco. Todos têm `tenure = 0`: são clientes que acabaram de entrar e **ainda não foram cobrados**, então o valor correto é `0`. Aplicamos **uma única regra** (`fillna(0)`), sem `dropna` e sem tratamento duplicado, e mantemos os 7.043 clientes. Preencher com uma constante não depende dos dados, portanto não causa vazamento.
- Em `OnlineSecurity`, `TechSupport` etc., "No internet service" é redundante com `InternetService = No`; o mesmo vale para "No phone service" em `MultipleLines`. Unificamos em "No" para não duplicar colunas e facilitar a leitura da importância.

In [ ]:
df = df_raw.copy().set_index('customerID')

# TotalCharges: texto -> número. Os brancos viram NaN e depois 0 (clientes com tenure = 0, ainda sem cobrança).
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
brancos = df['TotalCharges'].isna()
print(f'TotalCharges vazios: {brancos.sum()}  |  valores de tenure nessas linhas: {df.loc[brancos, "tenure"].unique()}')
df['TotalCharges'] = df['TotalCharges'].fillna(0.0)

# Categorias redundantes
COLS_SERVICOS = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
                 'StreamingTV', 'StreamingMovies']
df[COLS_SERVICOS] = df[COLS_SERVICOS].replace('No internet service', 'No')
df['MultipleLines'] = df['MultipleLines'].replace('No phone service', 'No')

# Alvo: Yes/No -> 1/0
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

assert df.isna().sum().sum() == 0, 'Ainda existem valores nulos'
print(f'\nBase limpa: {df.shape[0]} clientes x {df.shape[1]} colunas  |  taxa de churn: {df["Churn"].mean():.2%}')

### 1.3 Engenharia de features
Quatro variáveis criadas **linha a linha** (não usam estatísticas da base, logo não há vazamento) e ligadas aos padrões de risco que a EDA já mostrou:

| Feature | Ideia |
|---|---|
| `n_servicos_extras` | Quantos serviços adicionais o cliente tem (segurança, backup, proteção, suporte, streaming). Mais serviços = mais "amarras" |
| `fibra_sem_suporte` | Fibra óptica **sem** suporte técnico (combinação de risco conhecida) |
| `mensal_cheque_eletronico` | Contrato mensal **e** cheque eletrônico (baixo compromisso + pagamento manual) |
| `cliente_novo_12m` | Cliente com até 12 meses de casa (fase mais vulnerável) |

In [ ]:
def criar_features(d):
    d = d.copy()
    d['n_servicos_extras'] = (d[COLS_SERVICOS] == 'Yes').sum(axis=1)
    d['fibra_sem_suporte'] = ((d['InternetService'] == 'Fiber optic') & (d['TechSupport'] == 'No')).astype(int)
    d['mensal_cheque_eletronico'] = ((d['Contract'] == 'Month-to-month') &
                                     (d['PaymentMethod'] == 'Electronic check')).astype(int)
    d['cliente_novo_12m'] = (d['tenure'] <= 12).astype(int)
    return d


X = criar_features(df.drop(columns='Churn'))
y = df['Churn']

# Grupos de colunas para o pré-processamento
COLS_NUM = ['tenure', 'MonthlyCharges', 'TotalCharges', 'n_servicos_extras']                 # padronizar
COLS_BIN = ['SeniorCitizen', 'fibra_sem_suporte', 'mensal_cheque_eletronico', 'cliente_novo_12m']  # já são 0/1
COLS_CAT = [c for c in X.columns if c not in COLS_NUM + COLS_BIN]                            # one-hot
print(f'{len(COLS_NUM)} numéricas | {len(COLS_BIN)} binárias | {len(COLS_CAT)} categóricas')

### 1.4 Separação treino/teste (estratificada)
O conjunto de teste fica **trancado** até a avaliação final. `stratify=y` garante a mesma proporção de churn nos dois lados.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED)

print(f'Treino: {X_train.shape[0]} linhas ({y_train.mean():.2%} churn)')
print(f'Teste : {X_test.shape[0]} linhas ({y_test.mean():.2%} churn)')

### 1.5 Pré-processador sem vazamento de dados
`StandardScaler` e `OneHotEncoder` ficam dentro de um `ColumnTransformer`, que por sua vez vai dentro do `Pipeline` de cada modelo. Dessa forma:
- no `fit`, médias, desvios e categorias são aprendidos **somente nos dados de treino** (e, na validação cruzada, somente no treino de cada fold);
- no teste (ou no fold de validação) só se aplica `transform`;
- categorias nunca vistas no treino são ignoradas (`handle_unknown='ignore'`), em vez de quebrar o código.

A célula abaixo faz uma verificação visual: o treino fica com média 0 e desvio 1 por construção; o teste **não**, e isso é exatamente o comportamento esperado.

In [ ]:
def construir_preprocessador():
    return ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), COLS_NUM),
            ('cat', OneHotEncoder(drop='if_binary', handle_unknown='ignore', sparse_output=False), COLS_CAT),
            ('bin', 'passthrough', COLS_BIN),
        ],
        verbose_feature_names_out=False)


prep_demo = construir_preprocessador().fit(X_train)       # aprende só com o treino
Xtr_t, Xte_t = prep_demo.transform(X_train), prep_demo.transform(X_test)
n_num = len(COLS_NUM)

verificacao = pd.DataFrame({
    'média (treino)': Xtr_t[:, :n_num].mean(axis=0), 'desvio (treino)': Xtr_t[:, :n_num].std(axis=0),
    'média (teste)': Xte_t[:, :n_num].mean(axis=0), 'desvio (teste)': Xte_t[:, :n_num].std(axis=0),
}, index=COLS_NUM).round(3)
display(verificacao)
print(f'Matriz final: {Xtr_t.shape[1]} features após o encoding.')

### 1.6 EDA rápida: os padrões de risco que vamos "cobrar" do modelo
Calculada **só no treino**. Estas hipóteses voltam na seção 3.5, quando compararmos com a Feature Importance, e na seção 4, para segmentar os clientes.

In [ ]:
treino_eda = X_train.assign(Churn=y_train)
churn_geral = y_train.mean() * 100


def taxa_churn(base, coluna):
    return (base.groupby(coluna)['Churn'].mean() * 100).sort_values(ascending=False)


def barra_taxa(ax, serie, titulo, cor=COR_OK, rot=0):
    sns.barplot(x=serie.index.astype(str), y=serie.values, color=cor, ax=ax)
    ax.bar_label(ax.containers[0], fmt='%.1f%%', padding=2)
    ax.axhline(churn_geral, color='black', ls='--', lw=1, label=f'média geral ({churn_geral:.1f}%)')
    ax.set_title(titulo)
    ax.set_xlabel('')
    ax.set_ylabel('% de churn')
    ax.tick_params(axis='x', rotation=rot)
    ax.legend(loc='upper right', fontsize=9)


internet_suporte = (treino_eda['InternetService'].map({'Fiber optic': 'Fibra', 'DSL': 'DSL', 'No': 'Sem internet'})
                    + np.where(treino_eda['InternetService'] == 'No', '',
                               np.where(treino_eda['TechSupport'] == 'Yes', ' c/ suporte', ' s/ suporte')))
faixas_tenure = pd.cut(treino_eda['tenure'], bins=[-1, 6, 12, 24, 48, 72],
                       labels=['0-6m', '7-12m', '13-24m', '25-48m', '49-72m'])

fig, axes = plt.subplots(2, 2, figsize=(15, 9))
barra_taxa(axes[0, 0], taxa_churn(treino_eda, 'Contract'), 'Churn por tipo de contrato', COR_CHURN)
barra_taxa(axes[0, 1], taxa_churn(treino_eda.assign(grupo=internet_suporte), 'grupo'),
           'Churn por internet e suporte técnico', COR_CHURN, rot=20)
barra_taxa(axes[1, 0], taxa_churn(treino_eda, 'PaymentMethod'), 'Churn por forma de pagamento', COR_CHURN, rot=15)
barra_taxa(axes[1, 1], taxa_churn(treino_eda.assign(faixa=faixas_tenure), 'faixa').reindex(
    ['0-6m', '7-12m', '13-24m', '25-48m', '49-72m']), 'Churn por tempo de casa (tenure)', COR_CHURN)
fig.suptitle('EDA: onde o churn se concentra (base de treino)', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
salvar_fig('eda_churn_por_segmento')
plt.show()

---
## 2. Métricas adequadas e desbalanceamento de classes

### 2.1 Por que a acurácia engana
Com ~26,5% de churn, um "modelo" que responde **sempre "não vai cancelar"** já acerta ~73,5% e **não identifica nenhum** cliente em risco. Veja abaixo.

In [ ]:
dummy = DummyClassifier(strategy='most_frequent').fit(np.zeros((len(y_train), 1)), y_train)
pred_dummy = dummy.predict(np.zeros((len(y_test), 1)))
print('Modelo "sempre diz que o cliente fica":')
print(f'  Acurácia : {accuracy_score(y_test, pred_dummy):.1%}   <- parece ótimo...')
print(f'  Recall   : {recall_score(y_test, pred_dummy):.1%}   <- ...mas não captura nenhum churner')
print(f'  Precision: {precision_score(y_test, pred_dummy, zero_division=0):.1%}')

### 2.2 Métricas que usaremos

| Métrica | Pergunta que responde | Por que importa aqui |
|---|---|---|
| **Recall** | Dos clientes que **realmente cancelariam**, quantos o modelo pegou? | Principal métrica de negócio: cada churner perdido é receita que vai embora |
| **Precision** | Dos clientes que o modelo **apontou**, quantos realmente cancelariam? | Mede o desperdício do orçamento de retenção (descontos e contatos) |
| **F1** | Média harmônica entre Precision e Recall | Equilíbrio entre os dois |
| **F2** | Como o F1, mas o Recall pesa **2×** mais | Reflete a assimetria de custos: perder um cliente custa muito mais que um contato desnecessário |
| **ROC-AUC** | Qualidade do **ranking** (independe do threshold) | Importante porque vamos *priorizar* clientes pela probabilidade |
| **PR-AUC** | Idem, mais sensível em classes desbalanceadas | Complementa o ROC-AUC |

**Por que destacar o Recall?** Reter é mais barato do que adquirir, e o custo de um falso negativo (cliente que sai sem ser abordado) é a margem de todos os meses que ele ficaria. O custo de um falso positivo é um contato e, no pior caso, um desconto. Na seção 4 vamos **quantificar** essa assimetria com números.

In [ ]:
def metricas_limiar(y_true, proba, limiar=0.5):
    """Métricas que dependem do threshold."""
    pred = (np.asarray(proba) >= limiar).astype(int)
    return {'Precision': precision_score(y_true, pred, zero_division=0),
            'Recall': recall_score(y_true, pred),
            'F1': f1_score(y_true, pred),
            'F2': fbeta_score(y_true, pred, beta=2)}


def metricas_completas(y_true, proba, limiar=0.5):
    return {**metricas_limiar(y_true, proba, limiar),
            'ROC-AUC': roc_auc_score(y_true, proba),
            'PR-AUC': average_precision_score(y_true, proba)}


# Validação cruzada estratificada, usada em TODAS as decisões (só no treino)
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
SCORING = {'precision': 'precision', 'recall': 'recall', 'f1': 'f1',
           'f2': make_scorer(fbeta_score, beta=2), 'roc_auc': 'roc_auc', 'pr_auc': 'average_precision'}
NOMES_METRICAS = {'precision': 'Precision', 'recall': 'Recall', 'f1': 'F1',
                  'f2': 'F2', 'roc_auc': 'ROC-AUC', 'pr_auc': 'PR-AUC'}

neg, pos = np.bincount(y_train)
SPW = neg / pos        # razão negativos/positivos, usada no scale_pos_weight do XGBoost

### 2.3 Construtor de modelos com três estratégias para o desbalanceamento

| Técnica | Como funciona |
|---|---|
| `nenhuma` | Referência (baseline) |
| `class_weight` | Penaliza mais o erro na classe minoritária (`class_weight='balanced_subsample'` no RF, `scale_pos_weight` no XGBoost, `auto_class_weights='Balanced'` no CatBoost) |
| `smote` | Cria clientes sintéticos da classe churn **apenas nos dados de treino** de cada fold (SMOTE dentro do `Pipeline`, nunca no teste) |

> **Nota sobre o SMOTE:** ele interpola vizinhos no espaço das features, e após o one-hot as colunas categóricas ficam com valores fracionários nos pontos sintéticos. Para esta base é uma aproximação prática (o `SMOTENC` seria a alternativa purista). Quem decide se vale a pena é a validação cruzada.

In [ ]:
def construir_modelo(nome, tecnica):
    """Retorna um Pipeline: pré-processamento -> [SMOTE] -> modelo.
    nome: 'RandomForest' | 'XGBoost' | 'CatBoost'  |  tecnica: 'nenhuma' | 'class_weight' | 'smote'
    """
    peso = tecnica == 'class_weight'
    if nome == 'RandomForest':
        clf = RandomForestClassifier(n_estimators=300, min_samples_leaf=3,
                                     class_weight='balanced_subsample' if peso else None,
                                     n_jobs=1, random_state=SEED)
    elif nome == 'XGBoost':
        clf = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8,
                            colsample_bytree=0.8, scale_pos_weight=SPW if peso else 1.0,
                            eval_metric='logloss', n_jobs=1, random_state=SEED)
    elif nome == 'CatBoost':
        clf = CatBoostClassifier(iterations=300, depth=5, learning_rate=0.05,
                                 auto_class_weights='Balanced' if peso else None,
                                 verbose=0, random_seed=SEED, thread_count=1, allow_writing_files=False)
    else:
        raise ValueError(f'Modelo desconhecido: {nome}')

    etapas = [('prep', construir_preprocessador())]
    if tecnica == 'smote':
        etapas.append(('smote', SMOTE(random_state=SEED)))
    etapas.append(('model', clf))
    return Pipeline(etapas)


cache_cv = {}      # evita recalcular combinações já avaliadas


def avaliar_cv(nome, tecnica):
    chave = (nome, tecnica)
    if chave not in cache_cv:
        res = cross_validate(construir_modelo(nome, tecnica), X_train, y_train,
                             cv=CV, scoring=SCORING, n_jobs=N_JOBS)
        linha = {NOMES_METRICAS[m]: res[f'test_{m}'].mean() for m in SCORING}
        linha['ROC-AUC (desvio)'] = res['test_roc_auc'].std()
        cache_cv[chave] = linha
    return cache_cv[chave]

### 2.4 Teste das técnicas de balanceamento (Random Forest, validação cruzada 5-fold no treino)
Os valores abaixo usam o threshold padrão de 0,50. É esperado que `class_weight` e `smote` **aumentem o Recall** (à custa de Precision), enquanto o ROC-AUC, que mede só o ranking, muda pouco. Mais adiante o threshold será otimizado.

In [ ]:
tecnicas = ['nenhuma', 'class_weight', 'smote']
tab_balanceamento = pd.DataFrame({t: avaliar_cv('RandomForest', t) for t in tecnicas}).T
tab_balanceamento.index.name = 'técnica'
display(fmt_pct(tab_balanceamento, tab_balanceamento.columns))

base = tab_balanceamento.loc['nenhuma']
for t in ['class_weight', 'smote']:
    d = tab_balanceamento.loc[t] - base
    print(f'{t:>12} vs. nenhuma -> Recall {d["Recall"]*100:+.1f} p.p. | Precision {d["Precision"]*100:+.1f} p.p. '
          f'| F2 {d["F2"]*100:+.1f} p.p. | ROC-AUC {d["ROC-AUC"]*100:+.1f} p.p.')

graf = (tab_balanceamento[['Precision', 'Recall', 'F1', 'F2', 'ROC-AUC']].reset_index()
        .melt(id_vars='técnica', var_name='métrica', value_name='valor'))
fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(data=graf, x='métrica', y='valor', hue='técnica', palette=[COR_OK, COR_CHURN, COR_DESTAQUE], ax=ax)
for c in ax.containers:
    ax.bar_label(c, fmt='%.2f', padding=2, fontsize=8)
ax.set_ylim(0, 1.05)
ax.set_title('Random Forest: efeito do tratamento de desbalanceamento (CV 5-fold, threshold 0,50)')
ax.set_xlabel('')
ax.set_ylabel('valor médio na validação cruzada')
plt.tight_layout()
salvar_fig('comparacao_desbalanceamento_rf')
plt.show()

---
## 3. Modelagem avançada, otimização e Feature Importance

### 3.1 Comparação de algoritmos: Random Forest × XGBoost × CatBoost
Cada algoritmo é testado com as duas técnicas de balanceamento (`class_weight` e `smote`), totalizando 6 combinações avaliadas com **a mesma validação cruzada** no treino. Todos recebem a mesma matriz pré-processada, o que torna a comparação justa. O vencedor é escolhido pelo **ROC-AUC médio**, a medida de qualidade de ranking que independe do threshold, já que o threshold será otimizado depois.

In [ ]:
modelos = ['RandomForest', 'XGBoost', 'CatBoost']
tecnicas_cmp = ['class_weight', 'smote']

linhas = []
for m in modelos:
    for t in tecnicas_cmp:
        print(f'Avaliando {m} + {t} ...')
        linhas.append({'modelo': m, 'técnica': t, **avaliar_cv(m, t)})

tab_modelos = pd.DataFrame(linhas).set_index(['modelo', 'técnica']).sort_values('ROC-AUC', ascending=False)
display(fmt_pct(tab_modelos, tab_modelos.columns, casas=2))

MELHOR_MODELO, MELHOR_TECNICA = tab_modelos.index[0]
print(f'\nMelhor combinação (ROC-AUC na CV): {MELHOR_MODELO} + {MELHOR_TECNICA}  ->  {tab_modelos.iloc[0]["ROC-AUC"]:.4f}')

graf = tab_modelos.reset_index()
graf['combinação'] = graf['modelo'] + ' + ' + graf['técnica']
graf = graf.melt(id_vars='combinação', value_vars=['ROC-AUC', 'PR-AUC', 'Recall', 'F2'],
                 var_name='métrica', value_name='valor')
fig, ax = plt.subplots(figsize=(13, 5.5))
sns.barplot(data=graf, x='métrica', y='valor', hue='combinação', palette='tab10', ax=ax)
ax.set_ylim(0, 1.0)
ax.set_title('Comparação de modelos e técnicas de balanceamento (CV 5-fold no treino)')
ax.set_xlabel('')
ax.set_ylabel('valor médio')
ax.legend(loc='lower center', ncol=3, fontsize=9, bbox_to_anchor=(0.5, -0.32))
plt.tight_layout()
salvar_fig('comparacao_modelos')
plt.show()

### 3.2 Otimização de hiperparâmetros (RandomizedSearchCV)
Otimizamos **somente o melhor modelo** encontrado, preservando a mesma técnica de balanceamento. A busca roda dentro do `Pipeline`, então scaler, encoder e SMOTE são reajustados em cada fold, sem vazamento. O critério de refit é o ROC-AUC, e as demais métricas ficam registradas para consulta.

In [ ]:
PARAM_DISTS = {
    'RandomForest': {
        'model__n_estimators': [200, 300, 500],
        'model__max_depth': [4, 6, 8, 12, None],
        'model__min_samples_leaf': [1, 3, 5, 10],
        'model__max_features': ['sqrt', 0.3, 0.5],
    },
    'XGBoost': {
        'model__n_estimators': [200, 300, 500],
        'model__max_depth': [3, 4, 5, 6],
        'model__learning_rate': [0.01, 0.03, 0.05, 0.1],
        'model__subsample': [0.7, 0.8, 1.0],
        'model__colsample_bytree': [0.6, 0.8, 1.0],
        'model__min_child_weight': [1, 3, 5],
        'model__gamma': [0, 0.1, 0.3],
    },
    'CatBoost': {
        'model__iterations': [200, 300, 500],
        'model__depth': [3, 4, 5, 6],
        'model__learning_rate': [0.01, 0.03, 0.05, 0.1],
        'model__l2_leaf_reg': [1, 3, 5, 9],
    },
}
dist = dict(PARAM_DISTS[MELHOR_MODELO])
if MELHOR_TECNICA == 'smote':
    dist['smote__k_neighbors'] = [3, 5, 7]

busca = RandomizedSearchCV(
    estimator=construir_modelo(MELHOR_MODELO, MELHOR_TECNICA),
    param_distributions=dist, n_iter=N_ITER_BUSCA,
    scoring=SCORING, refit='roc_auc', cv=CV,
    n_jobs=N_JOBS, random_state=SEED, verbose=1)
busca.fit(X_train, y_train)

print(f'\nMelhores hiperparâmetros: {busca.best_params_}')
print(f'ROC-AUC (CV)  baseline: {tab_modelos.iloc[0]["ROC-AUC"]:.4f}  ->  otimizado: {busca.best_score_:.4f}')

resultados = pd.DataFrame(busca.cv_results_).sort_values('rank_test_roc_auc').head(5)
colunas_busca = ['mean_test_roc_auc', 'mean_test_pr_auc', 'mean_test_recall', 'mean_test_f2']
display(resultados[colunas_busca].rename(columns=lambda c: c.replace('mean_test_', '')).round(4))

modelo_final = busca.best_estimator_       # Pipeline já reajustado em todo o treino

### 3.3 Avaliação no conjunto de teste (threshold padrão = 0,50)
Primeira (e única) vez em que o teste é tocado para medir o desempenho. A matriz de confusão mostra o trade-off que o threshold vai ajustar a seguir.

In [ ]:
proba_test = modelo_final.predict_proba(X_test)[:, 1]
pred_050 = (proba_test >= 0.5).astype(int)

print(f'Modelo final: {MELHOR_MODELO} + {MELHOR_TECNICA}\n')
print(classification_report(y_test, pred_050, target_names=['Permanece', 'Churn'], digits=3))
print('Métricas completas (teste, threshold 0,50):')
print({k: round(v, 4) for k, v in metricas_completas(y_test, proba_test, 0.5).items()})

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fpr, tpr, _ = roc_curve(y_test, proba_test)
axes[0].plot(fpr, tpr, color=COR_CHURN, lw=2.5, label=f'{MELHOR_MODELO} (AUC = {roc_auc_score(y_test, proba_test):.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='aleatório')
axes[0].set(title='Curva ROC (teste)', xlabel='Taxa de falsos positivos', ylabel='Recall (taxa de verdadeiros positivos)')
axes[0].legend(loc='lower right')

prec, rec, _ = precision_recall_curve(y_test, proba_test)
axes[1].plot(rec, prec, color=COR_OK, lw=2.5, label=f'PR-AUC = {average_precision_score(y_test, proba_test):.3f}')
axes[1].axhline(y_test.mean(), color='k', ls='--', lw=1, label=f'aleatório ({y_test.mean():.2f})')
axes[1].set(title='Curva Precision-Recall (teste)', xlabel='Recall', ylabel='Precision')
axes[1].legend(loc='upper right')

sns.heatmap(confusion_matrix(y_test, pred_050), annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[2],
            xticklabels=['Prevê: fica', 'Prevê: churn'], yticklabels=['Real: fica', 'Real: churn'])
axes[2].set_title('Matriz de confusão (threshold 0,50)')
plt.tight_layout()
salvar_fig('avaliacao_modelo_teste')
plt.show()

### 3.4 Otimização do threshold de decisão com F2
O corte de 0,50 é arbitrário. Para escolher o threshold **sem olhar o teste**, geramos probabilidades **out-of-fold** no treino (cada cliente é previsto por um modelo que não o viu) e varremos os thresholds buscando o maior **F2**. O F2 favorece o Recall, mas ainda penaliza o excesso de falsos positivos, então evita o extremo de "contatar todo mundo". O threshold escolhido é aplicado ao teste uma única vez.

In [ ]:
proba_oof = cross_val_predict(modelo_final, X_train, y_train, cv=CV, method='predict_proba', n_jobs=N_JOBS)[:, 1]

limiares = np.round(np.arange(0.05, 0.951, 0.01), 2)
curva = pd.DataFrame([metricas_limiar(y_train, proba_oof, t) for t in limiares], index=limiares)
curva.index.name = 'threshold'

THR_F2 = float(curva['F2'].idxmax())
THR_F1 = float(curva['F1'].idxmax())
rec85 = curva.index[curva['Recall'] >= 0.85]
THR_R85 = float(rec85.max()) if len(rec85) else THR_F2
print(f'Threshold que maximiza o F2 (out-of-fold no treino): {THR_F2:.2f}  (F2 = {curva["F2"].max():.3f})')

fig, ax = plt.subplots(figsize=(11, 5.5))
for col, cor in zip(['Precision', 'Recall', 'F1', 'F2'], [COR_OK, COR_CHURN, '#55A868', COR_DESTAQUE]):
    ax.plot(curva.index, curva[col], label=col, lw=2.5 if col == 'F2' else 1.8, color=cor)
ax.axvline(0.5, color='gray', ls=':', label='padrão (0,50)')
ax.axvline(THR_F2, color=COR_DESTAQUE, ls='--', label=f'ótimo F2 ({THR_F2:.2f})')
ax.set(title='Métricas × threshold (probabilidades out-of-fold no treino)', xlabel='threshold de decisão', ylabel='métrica')
ax.legend(loc='lower center', ncol=3)
plt.tight_layout()
salvar_fig('otimizacao_threshold_f2')
plt.show()

**Impacto no teste:** comparação entre o threshold padrão e alternativas. A coluna "Falsos positivos" mostra o custo de ampliar a captura (clientes contatados à toa), e "Churners perdidos" mostra o que escapou.

In [ ]:
def tabela_limiares(y_true, proba, mapa):
    linhas = []
    for nome, t in mapa.items():
        pred = (np.asarray(proba) >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
        linhas.append({'critério': nome, 'threshold': t, **metricas_limiar(y_true, proba, t),
                       'alertas': int(tp + fp), 'churners capturados': int(tp),
                       'churners perdidos (FN)': int(fn), 'falsos positivos': int(fp)})
    return pd.DataFrame(linhas).set_index('critério')


mapa_thr = {'Padrão 0,50': 0.5, 'Máx. F1': THR_F1, 'Máx. F2 (escolhido)': THR_F2, 'Recall >= 85%': THR_R85}
tab_thr = tabela_limiares(y_test, proba_test, mapa_thr)
display(fmt_pct(tab_thr, ['Precision', 'Recall', 'F1', 'F2']).round({'threshold': 2}))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (nome, t) in zip(axes, [('Padrão 0,50', 0.5), (f'Máx. F2 ({THR_F2:.2f})', THR_F2)]):
    cm = confusion_matrix(y_test, (proba_test >= t).astype(int))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                xticklabels=['Prevê: fica', 'Prevê: churn'], yticklabels=['Real: fica', 'Real: churn'])
    m = metricas_limiar(y_test, proba_test, t)
    ax.set_title(f'{nome}\nRecall {m["Recall"]:.1%} | Precision {m["Precision"]:.1%} | F2 {m["F2"]:.3f}')
plt.tight_layout()
salvar_fig('matrizes_confusao_thresholds')
plt.show()

### 3.5 Feature Importance e validação contra os insights da EDA
Duas visões complementares:
- **Importância nativa do modelo** (nas features já codificadas): mostra o que o algoritmo usou para dividir os dados.
- **Importância por permutação** (nas colunas originais, medida no **teste**, em queda de ROC-AUC): mostra quanto o desempenho cai quando uma coluna é embaralhada. Observação: variáveis correlacionadas (ex.: `Contract` e `mensal_cheque_eletronico`) dividem a importância entre si.

In [ ]:
nomes_features = modelo_final.named_steps['prep'].get_feature_names_out()
imp = pd.Series(modelo_final.named_steps['model'].feature_importances_, index=nomes_features)
imp = (imp / imp.sum()).sort_values(ascending=False)

perm = permutation_importance(modelo_final, X_test, y_test, scoring='roc_auc',
                              n_repeats=10, random_state=SEED, n_jobs=N_JOBS)
imp_perm = (pd.DataFrame({'media': perm.importances_mean, 'desvio': perm.importances_std}, index=X_test.columns)
            .sort_values('media', ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(17, 7))
top = imp.head(15).iloc[::-1]
axes[0].barh(top.index, top.values, color=COR_OK)
axes[0].set(title=f'Importância nativa ({MELHOR_MODELO}), top 15', xlabel='importância relativa')

topp = imp_perm.head(12).iloc[::-1]
axes[1].barh(topp.index, topp['media'], xerr=topp['desvio'], color=COR_CHURN, capsize=3)
axes[1].set(title='Importância por permutação (teste), top 12', xlabel='queda no ROC-AUC ao embaralhar a coluna')
plt.tight_layout()
salvar_fig('importancia_features')
plt.show()

**Convergência com a EDA.** Para cada hipótese levantada na seção 1.6 calculamos o churn do segmento (no treino), o *lift* sobre a média e a melhor posição, no ranking de importância nativa, das features relacionadas. Consideramos que o modelo "converge" com a EDA se alguma delas estiver no top 10.

In [ ]:
ranking_imp = imp.rank(ascending=False, method='min').astype(int)

hipoteses = [
    ('Contrato mensal',               X_train['Contract'].eq('Month-to-month'),
     ['Contract_Month-to-month', 'Contract_One year', 'Contract_Two year']),
    ('Fibra óptica',                  X_train['InternetService'].eq('Fiber optic'),
     ['InternetService_Fiber optic', 'InternetService_DSL', 'InternetService_No']),
    ('Fibra óptica sem suporte',      X_train['fibra_sem_suporte'].eq(1),
     ['fibra_sem_suporte', 'TechSupport_Yes']),
    ('Cheque eletrônico',             X_train['PaymentMethod'].eq('Electronic check'),
     ['PaymentMethod_Electronic check', 'mensal_cheque_eletronico']),
    ('Cliente novo (<= 12 meses)',    X_train['tenure'].le(12),
     ['tenure', 'cliente_novo_12m']),
    ('Sem segurança online',          X_train['OnlineSecurity'].eq('No'),
     ['OnlineSecurity_Yes']),
    ('Mensalidade alta (>= $70)',     X_train['MonthlyCharges'].ge(70),
     ['MonthlyCharges']),
]

linhas = []
for nome, mascara, feats in hipoteses:
    ok = [f for f in feats if f in ranking_imp.index]
    melhor = int(ranking_imp[ok].min()) if ok else np.nan
    taxa = y_train[mascara.values].mean()
    linhas.append({'hipótese da EDA': nome, 'clientes no segmento': int(mascara.sum()),
                   'churn no segmento': taxa, 'churn geral': y_train.mean(), 'lift': taxa / y_train.mean(),
                   'melhor posição na importância': melhor,
                   'converge (top 10)?': 'SIM' if melhor <= 10 else 'não'})
tab_conv = pd.DataFrame(linhas).set_index('hipótese da EDA')
display(fmt_pct(tab_conv, ['churn no segmento', 'churn geral']).round({'lift': 2}))
print(f'{(tab_conv["converge (top 10)?"] == "SIM").sum()} de {len(tab_conv)} hipóteses da EDA aparecem entre as 10 features mais importantes do modelo.')

---
## 4. Da predição à retenção: o valor de negócio

Até aqui medimos *qualidade estatística*. Agora transformamos o modelo em **decisão**: quem contatar, com qual oferta e qual o retorno esperado.

### 4.1 Premissas de negócio
> **Atenção:** os valores abaixo são **hipóteses de trabalho** (a base da Telco não traz custos nem taxas de sucesso). Substitua pelos números reais da operação e, idealmente, calibre as taxas de sucesso com um **teste A/B**. A estrutura da simulação permanece a mesma.

**Lógica financeira (por cliente contatado):**
- **Custo** = custo do contato + `taxa_sucesso` × custo da oferta. Supomos que a oferta é concedida a quem aceita, e que a taxa de aceite é a própria `taxa_sucesso`. Isso inclui o **desconto dado a quem não cancelaria** (custo de falso positivo).
- **Benefício** = só existe se o cliente **realmente cancelaria** (`churn_real = 1`): `taxa_sucesso` × margem dos próximos `HORIZONTE_MESES` meses.
- **ROI** = (benefício − custo) / custo.

Como usamos o desfecho real do conjunto de teste, é um **backtest**: não depende de a probabilidade do modelo estar calibrada.

In [ ]:
# ---------- Premissas gerais (EDITE) ----------
HORIZONTE_MESES = 12     # meses de receita futura protegidos se o cliente for retido
MARGEM = 0.60            # margem de contribuição sobre a mensalidade
CUSTO_CONTATO = 4.0      # custo operacional de cada contato do time de CS (ligação/WhatsApp/e-mail)

# ---------- Plano de ação por segmento (EDITE) ----------
# desconto_pct x MonthlyCharges x meses_desconto + custo_fixo = custo da oferta quando o cliente aceita
ACOES = {
    'A. Mensal + Fibra sem suporte': dict(
        acao='Oferecer 20% de desconto por 12 meses na migração do contrato mensal para anual + suporte técnico gratuito por 6 meses',
        desconto_pct=0.20, meses_desconto=12, custo_fixo=30.0, taxa_sucesso=0.35),
    'B. Mensal + Cheque eletrônico': dict(
        acao='Migrar para débito automático/cartão com crédito de $10 + proposta de contrato anual com 10% de desconto',
        desconto_pct=0.10, meses_desconto=12, custo_fixo=10.0, taxa_sucesso=0.30),
    'C. Cliente novo (<= 12 meses)': dict(
        acao='Onboarding proativo: 3 meses de suporte técnico gratuito + contatos do CS no 1º, 3º e 6º mês',
        desconto_pct=0.0, meses_desconto=0, custo_fixo=15.0, taxa_sucesso=0.25),
    'D. Demais perfis': dict(
        acao='Contato proativo do CS + cupom de 10% por 3 meses',
        desconto_pct=0.10, meses_desconto=3, custo_fixo=0.0, taxa_sucesso=0.20),
}
plano_acao = pd.DataFrame.from_dict(ACOES, orient='index')
plano_acao.index.name = 'segmento'
for c in ['desconto_pct', 'meses_desconto', 'custo_fixo', 'taxa_sucesso']:
    plano_acao[c] = plano_acao[c].astype(float)
display(plano_acao)

### 4.2 Probabilidade de churn por cliente e Lista de Prioridade de Risco
Cada cliente do teste recebe a **probabilidade de churn** (`prob_churn`) e a lista é ordenada do maior para o menor risco.

Duas observações importantes:
- Modelos treinados com balanceamento **inflam** as probabilidades (o modelo "enxerga" 50% de churn no treino). O ranking continua válido, mas a probabilidade literal não é calibrada. Por isso incluímos também `prob_churn_calibrada`, obtida por **calibração de Platt** (regressão logística sobre o logit da probabilidade bruta) ajustada nas probabilidades out-of-fold **do treino**. Ela preserva a ordem e pode ser lida como probabilidade real.
- Neste notebook a lista é gerada sobre o **teste** (clientes que o modelo nunca viu), o que permite o backtest. Em produção, o modelo é reajustado com toda a história e aplicado à base de clientes ativos.

In [ ]:
def logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p)).reshape(-1, 1)


# Calibração de Platt ajustada em out-of-fold do treino -> sem tocar no teste
calibrador = LogisticRegression(C=1e6).fit(logit(proba_oof), y_train)

lista = X_test.copy()
lista['churn_real'] = y_test                                          # só para backtest
lista['prob_churn'] = proba_test
lista['prob_churn_calibrada'] = calibrador.predict_proba(logit(proba_test))[:, 1]
lista = lista.sort_values('prob_churn', ascending=False)
lista.insert(0, 'ranking', np.arange(1, len(lista) + 1))
lista['percentil'] = lista['ranking'] / len(lista)
lista['faixa_risco'] = pd.cut(lista['percentil'], bins=[0, 0.10, 0.30, 0.60, 1.0],
                              labels=['1-Crítico (top 10%)', '2-Alto (10-30%)', '3-Médio (30-60%)', '4-Baixo (60-100%)'])
lista['acima_threshold_f2'] = lista['prob_churn'] >= THR_F2

print(f'Probabilidade média prevista (bruta)    : {lista["prob_churn"].mean():.1%}')
print(f'Probabilidade média prevista (calibrada): {lista["prob_churn_calibrada"].mean():.1%}')
print(f'Taxa real de churn no teste             : {lista["churn_real"].mean():.1%}')
print(f'Clientes acima do threshold F2 ({THR_F2:.2f})  : {lista["acima_threshold_f2"].sum()} ({lista["acima_threshold_f2"].mean():.1%} da base)')

por_faixa = lista.groupby('faixa_risco', observed=True).agg(
    clientes=('ranking', 'size'), prob_media=('prob_churn', 'mean'),
    churn_real=('churn_real', 'mean'), churners=('churn_real', 'sum'))
por_faixa['lift'] = por_faixa['churn_real'] / lista['churn_real'].mean()
display(fmt_pct(por_faixa, ['prob_media', 'churn_real']).round({'lift': 2}))

fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(data=lista.assign(real=lista['churn_real'].map({0: 'Permaneceu', 1: 'Cancelou'})),
             x='prob_churn', hue='real', bins=40, stat='density', common_norm=False,
             palette={'Permaneceu': COR_OK, 'Cancelou': COR_CHURN}, alpha=0.55, ax=ax)
ax.axvline(THR_F2, color='black', ls='--', label=f'threshold F2 ({THR_F2:.2f})')
ax.set(title='Distribuição da probabilidade de churn por desfecho real (teste)', xlabel='probabilidade de churn prevista')
ax.legend(loc='upper center')
plt.tight_layout()
salvar_fig('distribuicao_probabilidades')
plt.show()

### 4.3 Segmentação dos clientes em risco e plano de ação
As regras seguem os padrões encontrados na EDA e confirmados pela Feature Importance. São avaliadas **em ordem de prioridade** (o cliente cai no primeiro segmento que se aplica):

1. **A. Mensal + Fibra sem suporte:** contrato mensal, fibra óptica e sem suporte técnico
2. **B. Mensal + Cheque eletrônico:** contrato mensal pagando com cheque eletrônico
3. **C. Cliente novo:** até 12 meses de casa
4. **D. Demais perfis**

In [ ]:
def segmentar(d):
    mensal = d['Contract'] == 'Month-to-month'
    condicoes = [
        mensal & (d['InternetService'] == 'Fiber optic') & (d['TechSupport'] == 'No'),
        mensal & (d['PaymentMethod'] == 'Electronic check'),
        d['tenure'] <= 12,
    ]
    return np.select(condicoes, ['A. Mensal + Fibra sem suporte', 'B. Mensal + Cheque eletrônico',
                                 'C. Cliente novo (<= 12 meses)'], default='D. Demais perfis')


lista['segmento'] = segmentar(lista)
lista['taxa_sucesso'] = lista['segmento'].map(plano_acao['taxa_sucesso'])
lista['custo_oferta'] = (lista['segmento'].map(plano_acao['desconto_pct']) * lista['MonthlyCharges']
                         * lista['segmento'].map(plano_acao['meses_desconto'])
                         + lista['segmento'].map(plano_acao['custo_fixo']))
lista['valor_em_risco'] = lista['MonthlyCharges'] * HORIZONTE_MESES * MARGEM
# A ação só é prescrita para quem está acima do threshold F2; os demais ficam em monitoramento
lista['acao_recomendada'] = np.where(lista['acima_threshold_f2'],
                                     lista['segmento'].map(plano_acao['acao']), 'Monitorar (sem ação ativa)')

em_risco = lista[lista['acima_threshold_f2']]
perfil = em_risco.groupby('segmento').agg(
    clientes=('ranking', 'size'), prob_media=('prob_churn', 'mean'), churn_real=('churn_real', 'mean'),
    mensalidade_media=('MonthlyCharges', 'mean'), valor_em_risco=('valor_em_risco', 'sum'))
perfil['% dos em risco'] = perfil['clientes'] / perfil['clientes'].sum()
perfil['ação'] = perfil.index.map(plano_acao['acao'])
display(fmt_pct(perfil, ['prob_media', 'churn_real', '% dos em risco']).round({'mensalidade_media': 2, 'valor_em_risco': 0}))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
ordem = perfil.index.tolist()
sns.barplot(x=perfil['clientes'], y=ordem, color=COR_OK, ax=axes[0])
axes[0].bar_label(axes[0].containers[0], padding=3)
axes[0].set(title='Clientes em risco por segmento (acima do threshold F2)', xlabel='clientes', ylabel='')
sns.barplot(x=perfil['churn_real'] * 100, y=ordem, color=COR_CHURN, ax=axes[1])
axes[1].bar_label(axes[1].containers[0], fmt='%.1f%%', padding=3)
axes[1].axvline(lista['churn_real'].mean() * 100, color='black', ls='--', lw=1, label='média geral')
axes[1].set(title='Churn real observado no segmento (backtest)', xlabel='% de churn', ylabel='')
axes[1].legend(loc='lower right')
plt.tight_layout()
salvar_fig('segmentos_risco')
plt.show()

### 4.4 Top 15 da Lista de Prioridade de Risco

In [ ]:
colunas_vista = ['ranking', 'prob_churn', 'prob_churn_calibrada', 'faixa_risco', 'segmento', 'Contract',
                 'InternetService', 'TechSupport', 'PaymentMethod', 'tenure', 'MonthlyCharges', 'acao_recomendada']
vista = lista[colunas_vista].head(15).copy()
vista[['prob_churn', 'prob_churn_calibrada']] = vista[['prob_churn', 'prob_churn_calibrada']].round(3)
pd.set_option('display.max_colwidth', 70)
display(vista)

### 4.5 Simulação de impacto financeiro e ROI por percentil de risco
Comparamos contatar apenas os clientes mais arriscados (top 5%, 10%, 20%...) com a **base inteira**. Para medir o valor real do modelo, cada cenário também traz o **lucro esperado se o mesmo número de clientes fosse escolhido ao acaso** (sem modelo). Como o resultado é linear nos clientes incluídos, esse valor esperado é exatamente `fração contatada × lucro da base inteira`.

A coluna `lift` indica quantas vezes a taxa de churn do grupo supera a da base.

**Como ler:** o ROI cai à medida que ampliamos o público, porque os clientes de risco menor custam quase o mesmo para contatar e devolvem menos. O lucro absoluto sobe e depois estabiliza, no ponto em que o cliente marginal deixa de pagar a própria oferta. O corte ideal, portanto, depende do **orçamento e da capacidade operacional** do time de CS: com orçamento limitado, é a priorização do modelo que separa uma campanha eficiente de uma dispersa.

In [ ]:
def simular_roi(grupo, mult_sucesso=1.0):
    """Backtest financeiro do grupo contatado. mult_sucesso permite análise de sensibilidade."""
    ts = (grupo['taxa_sucesso'] * mult_sucesso).clip(upper=1.0)
    n = len(grupo)
    churners = int(grupo['churn_real'].sum())
    custo = n * CUSTO_CONTATO + (ts * grupo['custo_oferta']).sum()
    beneficio = (grupo['churn_real'] * ts * grupo['valor_em_risco']).sum()
    return {'clientes contatados': n,
            'churners capturados': churners,
            '% dos churners da base': churners / lista['churn_real'].sum(),
            'precision (churn no grupo)': churners / n if n else np.nan,
            'lift': (churners / n) / lista['churn_real'].mean() if n else np.nan,
            'custo total ($)': custo,
            'receita retida ($)': beneficio,
            'lucro líquido ($)': beneficio - custo,
            'ROI': (beneficio - custo) / custo if custo else np.nan}


cenarios = {'Top 5%': 0.05, 'Top 10%': 0.10, 'Top 20%': 0.20, 'Top 30%': 0.30, 'Top 50%': 0.50,
            'Base inteira (100%)': 1.00}
linhas = []
for nome, frac in cenarios.items():
    n = int(np.ceil(frac * len(lista)))
    linhas.append({'cenário': nome, **simular_roi(lista.iloc[:n])})
linhas.append({'cenário': f'Acima do threshold F2 (>= {THR_F2:.2f})', **simular_roi(lista[lista['acima_threshold_f2']])})
roi_df = pd.DataFrame(linhas).set_index('cenário')

# Baseline sem modelo: mesmo volume de clientes escolhido ao acaso (valor esperado exato = fração x lucro da base inteira)
lucro_base = roi_df.loc['Base inteira (100%)', 'lucro líquido ($)']
roi_df['lucro se aleatório ($)'] = roi_df['clientes contatados'] / len(lista) * lucro_base
roi_df['ganho do modelo ($)'] = roi_df['lucro líquido ($)'] - roi_df['lucro se aleatório ($)']

exibir = fmt_pct(roi_df, ['% dos churners da base', 'precision (churn no grupo)', 'ROI'])
for c in ['custo total ($)', 'receita retida ($)', 'lucro líquido ($)', 'lucro se aleatório ($)', 'ganho do modelo ($)']:
    exibir[c] = roi_df[c].map(lambda v: f'${v:,.0f}')
display(exibir.round({'lift': 2}))

melhor_lucro = roi_df['lucro líquido ($)'].idxmax()
melhor_roi = roi_df['ROI'].idxmax()
print(f'Maior lucro líquido absoluto: {melhor_lucro} (${roi_df.loc[melhor_lucro, "lucro líquido ($)"]:,.0f})')
print(f'Maior ROI                   : {melhor_roi} ({roi_df.loc[melhor_roi, "ROI"]:.0%})')
print(f'Base inteira                : ROI {roi_df.loc["Base inteira (100%)", "ROI"]:.0%} | lucro líquido ${lucro_base:,.0f} | custo ${roi_df.loc["Base inteira (100%)", "custo total ($)"]:,.0f}')
for nome in ['Top 10%', 'Top 20%', 'Top 30%']:
    r = roi_df.loc[nome]
    print(f'{nome}: usa {r["custo total ($)"] / roi_df.loc["Base inteira (100%)", "custo total ($)"]:.0%} do orçamento da base inteira e '
          f'entrega {r["lucro líquido ($)"] / lucro_base:.0%} do lucro; '
          f'contatar o mesmo nº de clientes ao acaso renderia ${r["lucro se aleatório ($)"]:,.0f} '
          f'(o modelo multiplica por {r["lucro líquido ($)"] / r["lucro se aleatório ($)"]:.1f}x)')

# ---- Gráfico 1: lucro líquido e ROI por cenário ----
fig, ax1 = plt.subplots(figsize=(12, 5.8))
cores = [COR_OK if v >= 0 else COR_CHURN for v in roi_df['lucro líquido ($)']]
barras = ax1.bar(roi_df.index, roi_df['lucro líquido ($)'], color=cores, label='Lucro líquido com o modelo')
ax1.scatter(range(len(roi_df)), roi_df['lucro se aleatório ($)'], marker='_', s=900, linewidths=3,
            color='black', zorder=5, label='Lucro esperado sem modelo (mesmo volume, aleatório)')
ax1.axhline(0, color='black', lw=0.8)
ax1.bar_label(barras, labels=[f'${v:,.0f}' for v in roi_df['lucro líquido ($)']], padding=3, fontsize=9)
ax1.set_ylabel('Lucro líquido da campanha ($)')
ax1.set_ylim(top=roi_df['lucro líquido ($)'].max() * 1.18)
ax1.set_xticks(range(len(roi_df)))
ax1.set_xticklabels(roi_df.index, rotation=20, ha='right')
ax2 = ax1.twinx()
ax2.plot(range(len(roi_df)), roi_df['ROI'] * 100, 'o-', color=COR_DESTAQUE, lw=2.5, label='ROI (%) - eixo direito')
ax2.set_ylabel('ROI (%)')
ax2.set_ylim(bottom=min(0, roi_df['ROI'].min() * 100 * 1.1))
ax2.grid(False)
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc='upper center', bbox_to_anchor=(0.5, -0.28), ncol=3, fontsize=9)
ax1.set_title('Impacto financeiro da retenção por percentil de risco (backtest no teste)')
plt.tight_layout()
salvar_fig('simulacao_roi')
plt.show()

# ---- Gráfico 2: curva de ganho acumulado ----
x = np.r_[0, lista['percentil'].values]
y_ganho = np.r_[0, lista['churn_real'].cumsum().values / lista['churn_real'].sum()]
ideal = np.r_[0, np.minimum(np.arange(1, len(lista) + 1) / lista['churn_real'].sum(), 1.0)]
fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(x * 100, y_ganho * 100, color=COR_CHURN, lw=2.8, label='Modelo')
ax.plot(x * 100, ideal * 100, color='gray', ls=':', lw=1.5, label='Modelo perfeito')
ax.plot([0, 100], [0, 100], 'k--', lw=1, label='Sem modelo (aleatório)')
for p in [10, 20, 30]:
    v = np.interp(p / 100, x, y_ganho) * 100
    ax.scatter([p], [v], color=COR_DESTAQUE, zorder=5)
    ax.annotate(f'top {p}% -> {v:.0f}% dos churners', (p, v), textcoords='offset points', xytext=(12, -14), fontsize=9)
ax.set(title='Curva de ganho acumulado: % dos churners capturados × % da base contatada',
       xlabel='% da base contatada (ordenada por risco)', ylabel='% dos churners capturados')
ax.legend(loc='lower right')
plt.tight_layout()
salvar_fig('curva_ganho_acumulado')
plt.show()

### 4.6 Sensibilidade: e se a taxa de sucesso das ações for menor (ou maior) do que o suposto?
A taxa de sucesso é a premissa mais incerta. A tabela mostra o ROI de cada cenário multiplicando todas as taxas de sucesso por um fator. Onde o ROI cruza zero está o **ponto de equilíbrio** da campanha, e é o que devemos tentar validar primeiro em um teste A/B.

In [ ]:
multiplicadores = [0.50, 0.75, 1.00, 1.25]
sens = pd.DataFrame({f'Top {int(f*100)}%': {f'sucesso x{mult:.2f}': simular_roi(lista.iloc[:int(np.ceil(f * len(lista)))], mult)['ROI']
                                              for mult in multiplicadores}
                     for f in [0.10, 0.20, 0.30, 0.50, 1.00]})
sens = sens.rename(columns={'Top 100%': 'Base inteira'})
display(sens.map(lambda v: f'{v:.0%}'))

# Por que o F2 (e não o F1)? Comparação do custo de um falso negativo vs. um falso positivo
custo_fn = lista.loc[lista['churn_real'] == 1, 'valor_em_risco'].mean()
custo_fp = CUSTO_CONTATO + (lista['taxa_sucesso'] * lista['custo_oferta']).mean()
print(f'\nCusto médio de um falso NEGATIVO (churner não abordado): ${custo_fn:,.0f}')
print(f'Custo médio de um falso POSITIVO (contato + desconto desnecessário): ${custo_fp:,.0f}')
print(f'Perder um churner custa ~{custo_fn / custo_fp:.1f}x mais do que abordar um cliente à toa -> '
      f'justifica priorizar Recall (F2, beta = 2).')

---
## 5. Salvamento dos outputs
As figuras já foram gravadas em `./outputs/` ao longo do notebook (função `salvar_fig`). Agora exportamos a **Lista de Prioridade de Risco**, as tabelas de apoio e o modelo treinado.

In [ ]:
colunas_export = ['ranking', 'prob_churn', 'prob_churn_calibrada', 'faixa_risco', 'acima_threshold_f2',
                  'segmento', 'acao_recomendada', 'custo_oferta', 'valor_em_risco',
                  'Contract', 'InternetService', 'TechSupport', 'PaymentMethod', 'tenure', 'MonthlyCharges',
                  'churn_real']
lista_export = lista[colunas_export].rename(columns={'custo_oferta': 'custo_oferta_se_aceitar',
                                                     'churn_real': 'churn_real_backtest'}).copy()
lista_export[['prob_churn', 'prob_churn_calibrada']] = lista_export[['prob_churn', 'prob_churn_calibrada']].round(4)
lista_export[['custo_oferta_se_aceitar', 'valor_em_risco']] = lista_export[['custo_oferta_se_aceitar', 'valor_em_risco']].round(2)

# utf-8-sig mantém os acentos corretos ao abrir o CSV no Excel
lista_export.to_csv(os.path.join(OUTPUT_DIR, 'lista_prioridade_risco.csv'), index_label='customerID', encoding='utf-8-sig')

# Tabelas de apoio e modelo
roi_df.to_csv(os.path.join(OUTPUT_DIR, 'simulacao_roi.csv'), encoding='utf-8-sig')
plano_acao.to_csv(os.path.join(OUTPUT_DIR, 'plano_de_acao.csv'), encoding='utf-8-sig')
tab_modelos.to_csv(os.path.join(OUTPUT_DIR, 'comparacao_modelos.csv'), encoding='utf-8-sig')
tab_thr.to_csv(os.path.join(OUTPUT_DIR, 'comparacao_thresholds.csv'), encoding='utf-8-sig')
imp.rename('importancia_relativa').to_csv(os.path.join(OUTPUT_DIR, 'importancia_features.csv'), encoding='utf-8-sig')
joblib.dump({'pipeline': modelo_final, 'threshold_f2': THR_F2,
             'calibrador_platt': calibrador,      # aplicar sobre logit(prob_bruta) (função logit definida na seção 4.2)
             'modelo': MELHOR_MODELO, 'tecnica': MELHOR_TECNICA},
            os.path.join(OUTPUT_DIR, 'modelo_churn_final.joblib'))

print('Conteúdo de ./outputs:')
for arquivo in sorted(os.listdir(OUTPUT_DIR)):
    print(f'  {arquivo:<42} {os.path.getsize(os.path.join(OUTPUT_DIR, arquivo)) / 1024:8.1f} KB')

### Resumo executivo

In [ ]:
m050 = tab_thr.loc['Padrão 0,50']
mf2 = tab_thr.loc['Máx. F2 (escolhido)']
top20 = roi_df.loc['Top 20%']
print('=' * 78)
print('RESUMO EXECUTIVO')
print('=' * 78)
print(f'Modelo final            : {MELHOR_MODELO} + {MELHOR_TECNICA} (hiperparâmetros otimizados)')
print(f'ROC-AUC no teste        : {roc_auc_score(y_test, proba_test):.3f}   |   PR-AUC: {average_precision_score(y_test, proba_test):.3f}')
print(f'Threshold 0,50          : Recall {m050["Recall"]:.1%} | Precision {m050["Precision"]:.1%} | F2 {m050["F2"]:.3f}')
print(f'Threshold F2 ({THR_F2:.2f})     : Recall {mf2["Recall"]:.1%} | Precision {mf2["Precision"]:.1%} | F2 {mf2["F2"]:.3f}')
print(f'Top 20% de maior risco  : captura {top20["% dos churners da base"]:.0%} dos churners (lift {top20["lift"]:.1f}x)')
print(f'                          lucro líquido ${top20["lucro líquido ($)"]:,.0f} | ROI {top20["ROI"]:.0%} '
      f'(ao acaso, mesmo volume: ${top20["lucro se aleatório ($)"]:,.0f})')
print(f'Base inteira (100%)     : lucro líquido ${roi_df.loc["Base inteira (100%)", "lucro líquido ($)"]:,.0f} | ROI {roi_df.loc["Base inteira (100%)", "ROI"]:.0%}')
print(f'Corte que maximiza o lucro absoluto: {melhor_lucro} | corte que maximiza o ROI: {melhor_roi}')
print('=' * 78)

# Compacta tudo e (no Colab) baixa o .zip
shutil.make_archive('outputs', 'zip', OUTPUT_DIR)
print('Arquivo compactado: outputs.zip')
try:
    from google.colab import files
    files.download('outputs.zip')
except ImportError:
    pass

---
## Conclusões e próximos passos

**O que o projeto entrega agora**
1. Um pipeline **sem vazamento de dados**, com decisões tomadas só no treino e teste preservado para a avaliação final.
2. Métricas coerentes com o problema (Recall, Precision, F1, F2, ROC-AUC, PR-AUC), em vez da acurácia.
3. Três algoritmos × duas técnicas de balanceamento, comparados pela mesma validação cruzada, com o melhor otimizado.
4. Threshold escolhido por F2 **sem olhar o teste** e justificado financeiramente.
5. Uma **lista priorizada** de clientes, com segmento, ação recomendada e simulação de ROI, pronta para o time de Customer Success.

**Limitações e cuidados**
- Custos, margem e taxas de sucesso da seção 4 são **premissas**. O resultado financeiro só vale depois de substituí-las por números reais.
- O modelo prevê **quem vai cancelar**, não **quem será convencido pela oferta**. Quem tem alta probabilidade de churn nem sempre é o mais "persuadível".
- O dataset é um retrato estático. Em produção é preciso reajustar o modelo periodicamente e monitorar *drift*.

**Próximos passos recomendados**
1. Rodar um **teste A/B** com grupo de controle (sem ação) para medir a taxa de sucesso real de cada oferta e recalibrar a seção 4.
2. Evoluir para **uplift modeling**, que estima o efeito incremental da ação por cliente.
3. Explicabilidade individual com **SHAP**, para o CS entender *por que* cada cliente está na lista.
4. Incluir variáveis comportamentais (chamados ao suporte, queda de uso, atrasos de pagamento), que costumam ganhar da demografia.
5. Operacionalizar: agendar a geração mensal da lista e o monitoramento de desempenho.